### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

# step-1 Split the text into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# step-2 Embed the sentences
embeddings = model.encode(sentences)

# step-3 threshold parameter controls chunk tightness
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk = [sentences[0]]

# step-4 semantic grouping based on threshold
for i in range(1, len(sentences)):
    similarity = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]
    
    # step-5 if similarity is greater than threshold, add sentence to current chunk
    if similarity >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
        
# step-6 add the last chunk
chunks.append(" ".join(current_chunk))

# step-7 print the semantic chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


### RAG Pipeline Modular Coding

In [2]:
from langchain.schema import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# Custom Semantic Chunker With Threshold
class ThresholdSematicChunker:
    
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold 

    
    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            similarity = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            
            if similarity >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
                

        chunks.append(". ".join(current_chunk) + ".")
        
        return chunks
    
    
    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

In [3]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content= sample_text)
print(doc)


# Chunking
chunker = ThresholdSematicChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
chunks



page_content='
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
'


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# VectorStore
embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(chunks,embedding)

retriever = vectorstore.as_retriever()

In [5]:
from langchain_core.prompts import PromptTemplate

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [6]:
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnableLambda, RunnableMap
from langchain_core.output_parsers import StrOutputParser

# llm
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)

# LCEL Chain With retrieval
rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is used as a framework for building applications with Large Language Models (LLMs) and provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


### Semantic chunker With Langchain

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# Custom semantic chunker class
class SimpleSemanticChunker:
    def __init__(self, embedding_model, threshold=0.9):
        self.embedding_model = embedding_model
        self.threshold = threshold

    def split_documents(self, docs):
        results = []
        for doc in docs:
            sentences = doc.page_content.split(".")
            embeddings = self.embedding_model.embed_documents(sentences)

            current_chunk = [sentences[0]]
            for i in range(1, len(sentences)):
                sim = cosine_similarity(
                    [embeddings[i - 1]], [embeddings[i]]
                )[0][0]
                if sim >= self.threshold:
                    current_chunk.append(sentences[i])
                else:
                    results.append(" ".join(current_chunk))
                    current_chunk = [sentences[i]]
            results.append(" ".join(current_chunk))
        return results

# 1. Load the documents
loader = TextLoader("langchain-intro.txt")
docs = loader.load()

# 2. Initialize embedding model
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Create the custom chunker
chunker = SimpleSemanticChunker(embedding, threshold=0.9)

# 4. Split the documents
chunks = chunker.split_documents(docs)

# 5. Print results
for i, chunk in enumerate(chunks):
    print(f"\n📌 Chunk {i+1}:\n{chunk}")



📌 Chunk 1:
LangChain is a framework for building applications with LLMs

📌 Chunk 2:

Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone

📌 Chunk 3:

You can create chains, agents, memory, and retrievers

📌 Chunk 4:

The Eiffel Tower is located in Paris

📌 Chunk 5:

France is a popular tourist destination

📌 Chunk 6:

